In [17]:
import os
import pandas as pd
import matplotlib as plt
import seaborn as sns


In [18]:
df_avito = pd.read_csv("Bronze/apparts_vente.csv")

In [19]:
df_avito = df_avito.drop(
    columns=[
        
        'Meublé', 'Chauffage', 'Concierge', 'category', 'Sécurité',
        'Terrasse', 'Cuisine équipée', 'Balcon', 'Parking',
        'Climatisation', 'Ascenseur','Frais de syndic / mois', 'Disponibilité', 'Standing',
        'Surface totale', 'Âge du bien', 'seller_name', 'seller_phone', 'image_count', 
        'Salons', 'Étage', "Type d'appartement", 'seller_type', 'image_url','price_value'
    ]
)

In [20]:
df_avito = df_avito.rename(columns={
    'Surface habitable': 'surface_m2',
    'list_time': 'list_date',
    'Chambres': 'rooms',
    'Condition': 'condition',
    'Salle de bain': 'bathrooms',
})

In [21]:
def to_number(col):
    return pd.to_numeric(col.astype(str).str.replace(r"\D", "", regex=True), errors="coerce")

df_avito["price"] = to_number(df_avito["price"])
df_avito["surface_m2"] = to_number(df_avito["surface_m2"])
df_avito["list_date"] = pd.to_datetime(df_avito["list_date"], errors="coerce", utc=True).dt.tz_localize(None)
df_avito["area"] = df_avito["area"].astype("object")

In [22]:
df_avito.info()

<class 'pandas.DataFrame'>
RangeIndex: 41571 entries, 0 to 41570
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   description  41570 non-null  str           
 1   price        32898 non-null  float64       
 2   title        41571 non-null  str           
 3   area         0 non-null      object        
 4   surface_m2   40779 non-null  float64       
 5   list_date    41571 non-null  datetime64[us]
 6   city         41571 non-null  str           
 7   rooms        40997 non-null  float64       
 8   condition    31721 non-null  str           
 9   bathrooms    40041 non-null  float64       
 10  url          41571 non-null  str           
dtypes: datetime64[us](1), float64(4), object(1), str(5)
memory usage: 3.5+ MB


In [23]:
df_mubwab = pd.read_csv('Bronze/mubawab_listings.csv')

In [24]:
import re
from urllib.parse import unquote

# === 1. Extract dates from image_names ===
def extract_date_from_images(img_names):
    if pd.isna(img_names):
        return None
    decoded = unquote(str(img_names))
    
    patterns = [
        r'WhatsApp.*?(\d{4})-(\d{2})-(\d{2})',
        r'IMG-(\d{4})(\d{2})(\d{2})',
        r'(\d{4})(\d{2})(\d{2})_\d{6}',
        r'Generated.*?(\d{4}).*?(\d{2}).*?(\d{2})',
        r'(\d{4})(\d{2})(\d{2})',
    ]
    
    for pat in patterns:
        m = re.search(pat, decoded, re.IGNORECASE)
        if m:
            try:
                groups = m.groups()
                if len(groups) == 3:
                    y, mth, d = map(int, groups)
                    if 2015 <= y <= 2027 and 1 <= mth <= 12 and 1 <= d <= 31:
                        return f"{y:04d}-{mth:02d}-{d:02d}"
            except:
                continue
    return None

df_mubwab['extracted_date'] = df_mubwab['image_names'].apply(extract_date_from_images)
df_mubwab['listing_date'] = pd.to_datetime(df_mubwab['extracted_date'], errors='coerce')

# === 2. Fill missing dates using ad_id ===
df_mubwab['ad_id_num'] = pd.to_numeric(df_mubwab['ad_id'], errors='coerce')
df_mubwab = df_mubwab.sort_values('ad_id_num').reset_index(drop=True)

df_mubwab['listing_date_filled'] = df_mubwab['listing_date'].copy()
df_mubwab['listing_date_filled'] = df_mubwab['listing_date_filled'].interpolate(
    method='linear', 
    limit_direction='both'
)

# Final date
df_mubwab['listing_date'] = df_mubwab['listing_date'].combine_first(df_mubwab['listing_date_filled'])

# === NEW: Boolean column ===
df_mubwab['is_extracted'] = df_mubwab['listing_date'].notna()



In [25]:
split = df_mubwab['location'].str.split(r'\s*,+\s*', n=1, expand=True, regex=True)

df_mubwab['area'] = split[0].str.strip()
df_mubwab['city'] = split[1].str.strip()

In [26]:
df_mubwab = df_mubwab.drop(
    columns=[
        'location', 'ad_id', 'listing_type', 'orientation', 'floor_type',
        'garden_m2', 'garage_spaces', 'features_text', 'phone',
        'agency_name', 'agency_url', 'image_names', 'page', 'extracted_date',
        'listing_date', 'ad_id_num', 'is_extracted', 'bedrooms'
        
    ]
)

In [27]:
df_mubwab = df_mubwab.rename(columns={
    'price_DH': 'price',
    'listing_date_filled': 'list_date',
})

In [28]:

df_mubwab["price"] = to_number(df_mubwab["price"])

In [29]:
df_mubwab.info()

<class 'pandas.DataFrame'>
RangeIndex: 16658 entries, 0 to 16657
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   title        16658 non-null  str           
 1   price        15103 non-null  float64       
 2   surface_m2   16096 non-null  float64       
 3   rooms        14130 non-null  float64       
 4   bathrooms    6015 non-null   float64       
 5   condition    5922 non-null   str           
 6   description  7458 non-null   str           
 7   url          16658 non-null  str           
 8   list_date    16658 non-null  datetime64[us]
 9   area         16658 non-null  str           
 10  city         16539 non-null  str           
dtypes: datetime64[us](1), float64(4), str(6)
memory usage: 1.4 MB


In [30]:
df_mubwab.sample(10)

,title,price,surface_m2,rooms,bathrooms,condition,description,url,list_date,area,city
14595,Apartment location 3 chambres la colline,NaN,114.0,5.0,NaN,Bon état / habitable,Appartement a louer vide résidence de solidari...,https://www.mubawab.ma/fr/a/8322752/apartment-...,2026-03-25 08:00:00.000000,La Siesta,Mohammedia
13891,Appart a vendre meublé a Victor Hugo Marrakech,1600000.0,79.0,3.0,NaN,Projet neuf,Situé dans l'un des quartiers les plus calmes ...,https://www.mubawab.ma/fr/a/8315070/appart-a-v...,2026-02-28 00:00:00.000000,Camp Al Ghoul,Marrakech
4791,Bel appartement à vendre à route de casa. 2 ch...,1650000.0,97.0,3.0,1.0,Bon état / habitable,Opportunité unique pour cet appartement à vend...,https://www.mubawab.ma/fr/a/8166065/bel-appart...,2024-09-13 10:17:08.571428,Boumesmar,Marrakech
15434,Vente d'un bel appartement à Boulevard Abdelkr...,1780000.0,117.0,4.0,NaN,Bon état / habitable,Découvrez cet appartement à vendre. Prix 1 750...,https://www.mubawab.ma/fr/a/8330811/vente-d-un...,2026-04-01 18:00:00.000000,Route Casablanca,Marrakech
8824,Appartement a location,NaN,56.0,5.0,1.0,NaN,NaN,https://www.mubawab.ma/fr/a/8249470/appartemen...,2025-11-08 06:00:00.000000,Dar Bouazza,Dar Bouazza
3558,Vend appartement à Administratif. 2 chambres a...,110000000.0,65.0,2.0,NaN,NaN,NaN,https://www.mubawab.ma/fr/a/8135615/vend-appar...,2025-05-03 00:00:00.000000,Administratif,Tanger
14344,Appartement en vente à Casablanca Finance City...,970000.0,42.0,1.0,1.0,Projet neuf,Agréable appartement à vendre. Prix 970 000 DH...,https://www.mubawab.ma/fr/a/8320021/appartemen...,2026-03-18 06:00:00.000000,Casablanca Finance City,Casablanca
14170,Appartement à vendre à Émile Zola casablanca,1240000.0,124.0,4.0,NaN,Bon état / habitable,"À vendre, spacieux appartement de 124 m² situé...",https://www.mubawab.ma/fr/a/8318272/appartemen...,2026-03-16 12:00:00.000000,Belvédère,Casablanca
6857,Immeuble à Vendre à Derb OMAR,4800000.0,1200.0,4.0,1.0,NaN,NaN,https://www.mubawab.ma/fr/a/8212476/immeuble-%...,2025-08-23 00:00:00.000000,Derb Omar,Casablanca
7148,Appartement a vendre a assilah surface 87 m²,1150000.0,87.0,3.0,1.0,NaN,NaN,https://www.mubawab.ma/fr/a/8218066/appartemen...,2025-09-26 00:00:00.000000,Asilah,Asilah


In [31]:
df_avito["listing_type"] = "sale"
df_mubwab["listing_type"] = "sale"

df_merged = pd.concat([df_avito, df_mubwab])

In [32]:
df_merged.to_csv("Silver/sales_clean.csv")

In [33]:
df_merged

,description,price,title,area,surface_m2,list_date,city,rooms,condition,bathrooms,url,listing_type
0,"Situé à Mohammedia, le projet résidentiel ODYS...",495000.0,ODYSSÉE – Studios & Appartements haut standing,NaN,38.0,2026-05-31 11:18:22,Mohammedia,1.0,Neuf,0.0,https://www.avito.ma/fr/centre_ville/apparteme...,sale
1,فرصة رائعة لشراء شقة في بير الرامي، القنيطرة. ...,770000.0,À vendre – Appartement haut standing à Bir Rami,NaN,85.0,2026-05-26 11:54:21,Kénitra,2.0,Bon état,2.0,https://www.avito.ma/fr/bir_rami/appartements/...,sale
2,"Studio meublé à vendre – Oasis, Casablanca 📍 ...",1250000.0,STUDIO à vendre meublé 56 m² avec piscine à Oasis,NaN,52.0,2026-05-24 20:58:31,Casablanca,1.0,Bon état,1.0,https://www.avito.ma/fr/oasis/appartements/STU...,sale
3,Appartement à vendre à El Jadida El Ghorba prè...,430000.0,Appartement à vendre avec apport,NaN,62.0,2026-05-27 05:40:42,El Jadida,2.0,Neuf,1.0,https://www.avito.ma/fr/autre_secteur/appartem...,sale
4,Bel appartement meublé situé dans une résidenc...,870000.0,Appt meublé à vendre Rés fermée et sécurisée H24,NaN,72.0,2026-05-31 05:04:04,Casablanca,2.0,Bon état,1.0,https://www.avito.ma/fr/belv%C3%A9d%C3%A8re/ap...,sale
...,...,...,...,...,...,...,...,...,...,...,...,...
16653,"À vendre, appartement de 69 m² situé au 2ème é...",1160000.0,"Appartement à Vendre de 69.0 m² à Dar Bouazza,...",Dar Bouazza,69.0,2026-04-29 00:00:00,Dar Bouazza,4.0,NaN,NaN,https://www.mubawab.ma/fr/a/8341836/appartemen...,sale
16654,Nouvelle annonce de Vente d'un appartement à C...,1850000.0,"Appartement à Vendre de 130.0 m² à Casablanca,...",Maârif,130.0,2026-04-29 00:00:00,Casablanca,3.0,NaN,NaN,https://www.mubawab.ma/fr/a/8341843/appartemen...,sale
16655,Nouvelle annonce de Vente d'un appartement à D...,1195000.0,"Appartement à Vendre de 94.0 m² à Casablanca, ...",Les Hôpitaux,94.0,2026-04-29 00:00:00,Casablanca,4.0,NaN,NaN,https://www.mubawab.ma/fr/a/8341844/appartemen...,sale
16656,Appartement de Standing à Vendre – Route de Ta...,1850000.0,Opportunité exceptionnelle à saisir sur Marrakech,Route de Tahanaout,80.0,2026-04-29 00:00:00,Marrakech,3.0,Projet neuf,NaN,https://www.mubawab.ma/fr/a/8341869/opportunit...,sale
